# PDF convolution and detector resolution

This notebook demonstrates the generic one-dimensional convolution layer. It is intended for reconstructed observables such as invariant mass, decay time or other discriminating variables. It is conceptually distinct from Square-Dalitz SCF migration.


In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    ConvolvedPDF1D, FactorizedDensity, Gaussian1D, GaussianResolution1D,
    Parameter, enable_x64,
)

enable_x64()


## True PDF and Gaussian detector response

The observed density is obtained from

$$g(x)=\frac{\int f(t)R(x\mid t)dt}{\int_{x_{\min}}^{x_{\max}}dx\int f(t)R(x\mid t)dt}.$$

The denominator matters because a finite fit window can lose events through resolution tails.


In [ ]:
true_mass = Gaussian1D(
    mean=5.279, sigma=0.015, low=5.20, high=5.36,
)
resolution = GaussianResolution1D(sigma=0.010)
reco_mass = ConvolvedPDF1D(
    true_mass, resolution,
    true_low=5.20, true_high=5.36,
    observed_low=5.20, observed_high=5.36,
    order=96,
)

mass = jnp.linspace(5.20, 5.36, 1200)
plt.figure(figsize=(8,5))
plt.plot(mass, true_mass(mass), label="true PDF")
plt.plot(mass, reco_mass(mass), label="after resolution convolution")
plt.xlabel(r"$m$ [GeV]")
plt.ylabel("normalized density")
plt.legend()
plt.show()


In [ ]:
integral = jnp.trapezoid(reco_mass(mass), mass)
print("Observed-window normalization:", float(integral))
print("Fraction retained before renormalization:", float(reco_mass.normalization()))


## Bias and fitted resolution parameters

Both the resolution width and bias can be `Parameter` objects and therefore evaluated at different fit parameter values.


In [ ]:
sigma_res = Parameter("mass_resolution.sigma", 0.010, bounds=(0.002,0.050))
bias_res = Parameter("mass_resolution.bias", 0.0, bounds=(-0.020,0.020))

floating_reco = ConvolvedPDF1D(
    true_mass,
    GaussianResolution1D(sigma=sigma_res, bias=bias_res),
    true_low=5.20, true_high=5.36,
    observed_low=5.20, observed_high=5.36,
    order=96,
)

narrow = floating_reco(mass, {"mass_resolution.sigma":0.006, "mass_resolution.bias":0.0})
broad = floating_reco(mass, {"mass_resolution.sigma":0.020, "mass_resolution.bias":0.004})

plt.figure(figsize=(8,5))
plt.plot(mass, narrow, label="sigma=6 MeV, bias=0")
plt.plot(mass, broad, label="sigma=20 MeV, bias=4 MeV")
plt.xlabel(r"$m$ [GeV]")
plt.ylabel("normalized density")
plt.legend()
plt.show()


## Use inside a factorized Dalitz + discriminant density

The convolved PDF follows the same callable interface as `Gaussian1D`, `Exponential1D` and `Histogram1D`, so it can be inserted directly into `FactorizedDensity`.


In [ ]:
example_mass = jnp.asarray([5.272,5.279,5.291])
base_dalitz_density = lambda pars: jnp.asarray([0.2,0.5,0.3])

combined = FactorizedDensity(
    base_density=base_dalitz_density,
    observables={"mass":example_mass},
    pdfs={"mass":reco_mass},
)
print(combined({}))


## What this does not replace

A Dalitz-resolution or SCF problem is generally two-dimensional and can move events between distant regions of the physical Dalitz plane. That should remain a migration-kernel problem (`SquareDalitzSCFMap` or a future generalized migration operator), not two independent 1D convolutions in $s_{ij}$.
